# Your own metric, plot and callback

A metric is a pydantic model with a `type` alias: hooks accumulate one
scenario's batches, `summarise` returns a `Summary` subclass. A plot declares
the summary type it `requires`, so pairing it with the wrong metric fails
before the run. A callback sees every scenario and isn't saved with the
definition — use one for progress, logging or writing results somewhere.

In [1]:
import polars as pl
from IPython.display import display
from decider.experiments import (Bars, Changes, Describe, Divergence, Experiment, Progress, RunStatus,
                                 ValueCounts)
from example_projects.experiments.credit_pipeline import build


def show(spec):
    """Draw a plot's Vega-Lite spec inline; JupyterLab and VS Code render it interactively."""
    display({"application/vnd.vegalite.v5+json": spec}, raw=True)


ENTRY = "example_projects.experiments.credit_pipeline:build"
pipeline = build()
df = pl.DataFrame({
    "income": [1000.0, 2000.0, 1500.0, 800.0],
    "debt": [250.0, 500.0, 700.0, 250.0],
})
df

shape: (4, 2)
┌────────┬───────┐
│ income ┆ debt  │
│ ---    ┆ ---   │
│ f64    ┆ f64   │
╞════════╪═══════╡
│ 1000.0 ┆ 250.0 │
│ 2000.0 ┆ 500.0 │
│ 1500.0 ┆ 700.0 │
│ 800.0  ┆ 250.0 │
└────────┴───────┘


In [2]:
import typing as t
from decider.experiments import Callback, Metric, Plot, Summary
from decider.experiments.plots.bars import VEGA_LITE


class Rate(Summary):
    approved: int
    records: int


class ApprovalRate(Metric):
    type: t.Literal["approval_rate"] = "approval_rate"
    summary: t.ClassVar = Rate

    def reset(self):
        self._approved = self._records = 0

    def on_batch_end(self, batch):
        self._approved += int(batch.output["approved"].sum())
        self._records += batch.output.height

    def summarise(self):
        return Rate(approved=self._approved, records=self._records)


class RateBars(Plot):
    type: t.Literal["rate_bars"] = "rate_bars"
    requires: t.ClassVar = Rate

    def render(self, summaries):
        rows = [{"scenario": name, "rate": s.approved / s.records} for name, s in summaries.items()]
        return {"$schema": VEGA_LITE, "data": {"values": rows}, "mark": "bar",
                "encoding": {"x": {"field": "scenario", "sort": None},
                             "y": {"field": "rate", "type": "quantitative", "axis": {"format": "%"}},
                             "tooltip": [{"field": "scenario"}, {"field": "rate", "format": ".0%"}]}}


class PrintBatches(Callback):
    def on_batch_end(self, batch):
        print(f"{batch.scenario}: records {batch.offset}-{batch.offset + batch.output.height - 1}")

In [3]:
exp = (Experiment("approval-rate", pipeline, df, entry=ENTRY, batch_size=2)
       .scenario("strict", params={"credit": {"approved": {"limit": 0.28}}})
       .metric(ApprovalRate())
       .plot(RateBars(metric="approval_rate"))
       .callback(PrintBatches()))
result = exp.run()
result.summaries["approval_rate"]

baseline: records 0-1
strict: records 0-1
baseline: records 2-3
strict: records 2-3
{'baseline': Rate(approved=2, records=4), 'strict': Rate(approved=1, records=4)}


In [4]:
show(result.plots["rate_bars"])

<VegaLite 5 chart>

The definition — scenarios, metrics and plots, custom ones included — saves
as JSON and loads back by each class's `type` alias, so a saved experiment
can run headlessly with `decider.experiments.run` (as long as the classes are
importable where it runs).

In [5]:
import tempfile
from pathlib import Path
from decider.experiments import ExperimentDef, run

path = exp.save(Path(tempfile.mkdtemp()) / "experiment.json")
print(path.read_text()[:400])
again = run(ExperimentDef.model_validate_json(path.read_text()), frame=df)
again.summaries["approval_rate"] == result.summaries["approval_rate"]

{
  "version": "2",
  "name": "approval-rate",
  "description": "",
  "flow": {
    "entry": "example_projects.experiments.credit_pipeline:build",
    "decider": ""
  },
  "revision": "HEAD",
  "input": {
    "path": ""
  },
  "scenarios": [
    {
      "name": "strict",
      "revision": null,
      "params": {
        "credit": {
          "approved": {
            "limit": 0.28
          }
    
True
